# Figure 4 — IG-attribution "hero" spectrum

Re-renders the InstaNovo-FM integrated-gradients (IG) attribution **hero** figure
for **`KIEGNLIFDPNNYLPK`** (masked `y6+`), HCD / Orbitrap Fusion, directly from
the companion JSON — no model checkpoint or GPU required.

### Input
| Variable | File | Content |
|---|---|---|
| `report` | `data/hero_010_KIEGNLIFDPNNYLPK_y6+.json` | Per-peak data behind every panel |

### Reference code
This notebook re-implements the eval's plotting routine
(`instanovo…ig_attribution_helper.plot_hero_spectrum_attribution`, used by
`make_hero_attribution_figure.py`) so it runs standalone in the `instafm`
environment, in the same publication style as Figures 1 & 3. The lightweight
array/object reconstruction (`build_arrays`, `build_result`, `title_from`) is
taken verbatim from `make_hero_attribution_figure.py`.

### The four panels
1. **Index-based ion view** — peaks coloured by ion category; masked `y6+` in red.
2. **IG attribution** — per-peak integrated-gradients attribution for the masked group.
3. **Per-peak confidence strip** — model confidence at each visible peak.
4. **Bottom row** — (a) per-peak prediction confidence for the masked group;
   (b) ranked attribution by fragment group, coloured by relationship category.

Outputs are written as **.svg** into `figures/4/`.


In [ ]:
# ── Environment check — do not pip install from inside the notebook ───────────
# The environment is pinned by uv.lock, which is the source of truth: the
# constraints in pyproject.toml are floors, and the lock records exactly what
# resolved. Create it with ./setup_kernel.sh and select the resulting kernel.
#
# The pins matter because figure_3's zoom-region search ranks candidate windows
# whose separation scores frequently tie. Those rankings are total orders now,
# so the selection is reproducible across numpy versions -- this check guards
# against the rest of the stack drifting. Installing from a cell cannot fix a
# mismatch anyway: once numpy is imported, `pip install numpy==x` does not
# change the module already loaded in this kernel.
import importlib.metadata as _im
import pathlib as _pl
import warnings as _w

try:
    import tomllib as _tomllib
except ModuleNotFoundError:            # Python 3.10
    _tomllib = None

_root = next((p for p in (_pl.Path.cwd(), *_pl.Path.cwd().parents)
              if (p / 'uv.lock').is_file()), None)

# Packages that change what the figures show, or how they are laid out.
_WATCHED = ('numpy', 'pandas', 'pyarrow', 'scikit-learn', 'matplotlib', 'plotly')

if _root is None:
    _w.warn('uv.lock not found; skipping the environment check.')
elif _tomllib is None:
    _w.warn('tomllib needs Python 3.11+; skipping the environment check. '
            'setup_kernel.sh builds a 3.11 environment by default.')
else:
    # uv.lock can hold several resolution splits (requires-python spans 3.10-3.13),
    # so one package may be locked at more than one version. Any of them is valid.
    _locked = {}
    for _p in _tomllib.loads((_root / 'uv.lock').read_text()).get('package', []):
        _locked.setdefault(_p['name'], set()).add(_p['version'])

    _bad = []
    for _pkg in _WATCHED:
        _want = _locked.get(_pkg)
        if not _want:
            continue
        try:
            _got = _im.version(_pkg)
        except _im.PackageNotFoundError:
            _bad.append((_pkg, '/'.join(sorted(_want)), 'not installed'))
            continue
        if _got not in _want:
            _bad.append((_pkg, '/'.join(sorted(_want)), _got))

    if _bad:
        _msg = '\n'.join(f'  {p:16s} lock {w:12s} have {g}' for p, w, g in _bad)
        if any(p == 'numpy' for p, _, _ in _bad):
            raise RuntimeError(
                'Environment does not match uv.lock:\n' + _msg +
                '\n\nRun ./setup_kernel.sh and select the "InstaNovo-FM (figures)" kernel.')
        _w.warn('Environment differs from uv.lock:\n' + _msg)
    else:
        print(f'Environment matches uv.lock ({len(_WATCHED)} watched packages).')


In [ ]:
# ── Imports & paths ──────────────────────────────────────────────────────────
import json
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Repo root, independent of where the kernel was started ───────────────────
# Path.cwd() only worked when Jupyter happened to be launched from the repo root;
# with the notebooks in notebooks/ that silently pointed everything one level down.
# Walk up instead, anchored on files that only exist at the root.
def _find_repo_root(start=None):
    _p = (start or Path.cwd()).resolve()
    for _cand in (_p, *_p.parents):
        if (_cand / 'config' / 'metadata_colors.json').is_file() and (_cand / 'data').is_dir():
            return _cand
    raise RuntimeError(
        'Repo root not found: expected an ancestor holding config/metadata_colors.json '
        f'and data/. Searched upward from {_p}.')

BASE_DIR = _find_repo_root()
JSON_PATH = BASE_DIR / 'data' / 'hero_010_KIEGNLIFDPNNYLPK_y6+.json'

if not JSON_PATH.exists():
    raise FileNotFoundError(f'Missing input file: {JSON_PATH}\nWorking directory: {BASE_DIR}')

report = json.loads(JSON_PATH.read_text())
print(f'Working directory : {BASE_DIR}')
print(f'Loaded            : {JSON_PATH.name}')
print(f'Sequence          : {report["spectrum"]["sequence"]} | masked {report["masked_group"]["group_key"]}')
print(f'Visible peaks     : {len(report["visible_peaks"])} | ranked groups: {len(report["attribution"]["ranked_groups"])}')


In [ ]:
# ── Publication style (consistent with Figures 1 & 3) ────────────────────────
def set_publication_style():
    sns.set_theme(style="ticks")
    plt.rcParams.update({
        "font.family":      "serif",
        "font.serif":       ["Palatino", "Palatino Linotype", "TeX Gyre Pagella",
                             "Book Antiqua", "URW Palladio L", "DejaVu Serif"],
        "font.size":        14,
        "axes.titlesize":   16,
        "axes.labelsize":   15,
        "xtick.labelsize":  12,
        "ytick.labelsize":  12,
        "axes.linewidth":    1.5,
        "xtick.major.width": 1,
        "ytick.major.width": 1,
        "axes.spines.top":   False,
        "axes.spines.right": False,
        "figure.dpi":       300,
        "figure.facecolor": "white",
        "axes.facecolor":   "white",
        "legend.fontsize":      13,
        "legend.frameon":       False,
        "legend.columnspacing": 1.5,
        "axes.grid":      True,
        "grid.alpha":     0.25,
        "grid.color":     "#CCCCCC",
        "grid.linewidth": 0.5,
    })

set_publication_style()

FIGURES_DIR = BASE_DIR / 'figures' / '4'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

def save_fig(fig, name):
    path = str(FIGURES_DIR / name) + '.svg'
    fig.savefig(path, bbox_inches='tight')
    print(f'  Saved → figures/4/{name}.svg')

# ── Colour scheme ────────────────────────────────────────────────────────────
# Ion categories (panels 1-3) — colourblind-safe, masked group highlighted red.
ION_COLORS = {
    'y':        '#2E75B6',   # blue
    'b':        '#E69F00',   # orange
    'a':        '#009E73',   # green
    'c':        '#9467BD',   # purple
    'x':        '#56B4E9',
    'z':        '#CC79A7',
    'unannot':  '#BDBDBD',   # neutral grey
}
MASKED_COLOR = '#D62728'     # red — the masked group

# Relationship categories (panel 4b).
CATEGORY_COLORS = {
    'ladder_neighbor':     '#1F4E79',
    'near_ladder':         '#5B9BD5',
    'opposite_series':     '#E69F00',
    'distant_same_series': '#009E73',
    'internal_fragment':   '#9467BD',
    'other_annotated':     '#7F7F7F',
    'unannotated':         '#BDBDBD',
}
CATEGORY_LABELS = {
    'ladder_neighbor':     'Ladder neighbour',
    'near_ladder':         'Near ladder',
    'opposite_series':     'Opposite series',
    'distant_same_series': 'Distant same series',
    'internal_fragment':   'Internal fragment',
    'other_annotated':     'Other annotated',
    'unannotated':         'Unannotated',
}

def ion_category(annotation):
    if not annotation:
        return 'unannot'
    c = annotation[0].lower()
    return c if c in ION_COLORS else 'unannot'


In [ ]:
# ── Reconstruct arrays/objects from the JSON ─────────────────────────────────
# (taken from make_hero_attribution_figure.py — these rebuild the lightweight
#  objects the plotter expects from the JSON per-peak records).
_ISOTOPE_DA = 1.00335


def _charge_of(group_key: str) -> int:
    """Charge from a group key like 'y6+' / 'y13++' (number of trailing '+')."""
    return max(1, group_key.count("+"))


def build_arrays(report: dict):
    """Reconstruct full-length mz / intensity / attribution / confidence /
    annotation arrays from the JSON's per-peak records."""
    visible = report["visible_peaks"]
    mg = report["masked_group"]
    masked_indices = list(mg["peak_indices"])

    n = max(
        max((vp["peak_idx"] for vp in visible), default=-1),
        max(masked_indices, default=-1),
    ) + 1

    mz = np.zeros(n, dtype=float)
    intensity = np.zeros(n, dtype=float)
    attributions = np.zeros(n, dtype=float)
    confidence = np.zeros(n, dtype=float)
    annotations = [""] * n

    for vp in visible:
        i = vp["peak_idx"]
        mz[i] = vp["mz"]
        intensity[i] = vp["intensity"]
        attributions[i] = vp.get("attribution_for_this_group", 0.0)
        if vp.get("confidence") is not None:
            confidence[i] = vp["confidence"]
        if vp.get("annotation"):
            annotations[i] = vp["annotation"]

    # Masked group peaks are excluded from `visible_peaks`. Use their persisted
    # true processed m/z & intensity when present (exact re-render); otherwise
    # fall back to analytic m/z + placeholder intensity (median of visible).
    peak_mz = mg.get("peak_mz")
    peak_int = mg.get("peak_intensity")
    placeholder_int = float(np.median(intensity[intensity > 0])) if np.any(intensity > 0) else 0.5
    charge = _charge_of(mg["group_key"])
    base_idx = mg["base_idx"]
    peak_anns = mg.get("peak_annotations") or []
    for k, i in enumerate(masked_indices):
        if peak_mz and k < len(peak_mz):
            mz[i] = float(peak_mz[k])
        else:
            order = 0 if i == base_idx else 1  # treat non-base as the +1 isotope
            mz[i] = mg["base_mz"] + order * _ISOTOPE_DA / charge
        intensity[i] = float(peak_int[k]) if (peak_int and k < len(peak_int)) else placeholder_int
        if k < len(peak_anns) and peak_anns[k]:
            annotations[i] = peak_anns[k]

    return mz, intensity, attributions, confidence, annotations


def build_result(report: dict):
    """Rebuild the lightweight `AttributionResult`-like object the plotter reads."""
    mg = report["masked_group"]
    masked_group = SimpleNamespace(
        peak_indices=list(mg["peak_indices"]),
        base_idx=mg["base_idx"],
        group_key=mg["group_key"],
        base_mz=mg["base_mz"],
        ion_type=mg.get("ion_type", ""),
    )

    p = report["prediction"]
    true_mz = p.get("base_true_mz") or 0.0
    err_da = p.get("base_error_da") or 0.0
    err_ppm = (err_da / true_mz * 1e6) if true_mz else 0.0
    peak_predictions = [
        SimpleNamespace(
            annotation=pp.get("annotation", ""),
            true_mz_da=pp.get("true_mz", 0.0),
            predicted_mz_da=pp.get("predicted_mz", 0.0),
            log_prob=pp.get("log_prob", 0.0),
            correct_bin=bool(pp.get("correct_bin", False)),
            correct_bin_rank=pp.get("correct_bin_rank", -1) or -1,
        )
        for pp in p.get("per_peak", [])
    ]
    prediction = SimpleNamespace(
        correct_bin=bool(p.get("bin_correct", False)),
        group_bin_accuracy=p.get("group_bin_accuracy", -1.0),
        predicted_mz_da=p.get("base_predicted_mz", 0.0),
        true_mz_da=true_mz,
        error_da=err_da,
        error_ppm=err_ppm,
        confidence=p.get("mean_log_prob", 0.0),
        peak_predictions=peak_predictions,
    )

    ranked_peaks = [
        SimpleNamespace(
            annotation=rg.get("annotation", ""),
            attribution=rg.get("attribution_for_this_group", 0.0),
            category=rg.get("category", "unannotated"),
        )
        for rg in report["attribution"].get("ranked_groups", [])
    ]
    topk = SimpleNamespace(ranked_peaks=ranked_peaks)

    return SimpleNamespace(
        masked_group=masked_group,
        prediction=prediction,
        topk=topk,
        attributions=None,  # filled by caller (full-length array)
    )


def title_from(report: dict) -> str:
    s = report["spectrum"]
    mg = report["masked_group"]
    frag = s.get("fragmentation") or "?"
    inst = s.get("instrument") or "?"
    return (
        f"{s.get('sequence', '')} | z={s.get('precursor_charge', 0)} | "
        f"Masked: {mg['group_key']} (m/z={mg['base_mz']:.2f}) | {frag} | {inst}"
    )


mz, intensity, attributions, confidence, annotations = build_arrays(report)
result = build_result(report)
result.attributions = attributions
print(f'Reconstructed arrays of length {len(mz)} | title:')
print(' ', title_from(report))


In [ ]:
# ── Panel renderers (re-implementation of plot_hero_spectrum_attribution) ────
# Each renderer draws one panel onto a given Axes, so panels can be composed
# into the hero figure AND exported individually.
#
# Per request: only the first panel keeps a title. The IG-attribution and
# per-peak-confidence panels colour their bars as a heatmap by value, with a
# colourbar legend (as in the original eval figure).
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

ATTR_CMAP = 'magma'     # IG-attribution heatmap
CONF_CMAP = 'viridis'   # confidence heatmap


def _masked_set(result):
    return set(result.masked_group.peak_indices)


def panel_ion_view(ax, mz, intensity, annotations, result):
    """Panel 1 — index-based stem view, peaks coloured by ion category."""
    masked = _masked_set(result)
    idx = np.arange(len(intensity))
    visible = intensity > 0
    for i in idx[visible]:
        if i in masked:
            color, lw, z = MASKED_COLOR, 2.2, 5
        else:
            color, lw, z = ION_COLORS[ion_category(annotations[i])], 1.3, 3
        ax.vlines(i, 0, intensity[i], color=color, linewidth=lw, zorder=z)
    # label the masked base peak
    b = result.masked_group.base_idx
    ax.annotate(result.masked_group.group_key,
                xy=(b, intensity[b]), xytext=(0, 6), textcoords='offset points',
                ha='center', va='bottom', fontsize=11, fontweight='bold',
                color=MASKED_COLOR)
    ax.set_xlim(-2, len(intensity) + 1)
    ax.set_ylim(0, intensity.max() * 1.15)
    ax.set_xlabel('Peak index')
    ax.set_ylabel('Relative intensity')
    ax.set_title('Index-based ion view', loc='left', fontweight='bold')
    handles = [mpatches.Patch(color=MASKED_COLOR, label=f'masked {result.masked_group.group_key}')]
    for cat in ['y', 'b', 'a', 'c']:
        if any(ion_category(annotations[i]) == cat for i in idx[visible]):
            handles.append(mpatches.Patch(color=ION_COLORS[cat], label=f'{cat}-ion'))
    handles.append(mpatches.Patch(color=ION_COLORS['unannot'], label='unannotated'))
    ax.legend(handles=handles, ncol=2, loc='upper right', fontsize=10)


def panel_attribution(ax, attributions, result):
    """Panel 2 — per-peak IG attribution vs peak index, bars coloured (heatmap)
    by attribution magnitude with a colourbar. No panel title (per request); the
    prediction summary is kept as light metadata text."""
    masked = _masked_set(result)
    idx = np.arange(len(attributions))
    nz = np.abs(attributions) > 0
    vmax = float(np.max(attributions[nz])) if np.any(nz) else 1.0
    norm = Normalize(vmin=0.0, vmax=vmax)
    cmap = plt.get_cmap(ATTR_CMAP)
    bar_colors = [cmap(norm(attributions[i])) for i in idx[nz]]
    ax.bar(idx[nz], attributions[nz], width=0.9, color=bar_colors, zorder=3)
    # outline the masked group's own bars so they stay identifiable
    for i in idx[nz]:
        if i in masked:
            ax.bar(i, attributions[i], width=0.9, facecolor='none',
                   edgecolor=MASKED_COLOR, linewidth=1.4, zorder=4)
    ax.axhline(0, color='#555555', linewidth=0.8, zorder=2)
    ax.set_xlim(-2, len(attributions) + 1)
    ax.set_xlabel('Peak index')
    ax.set_ylabel('IG attribution')
    sm = ScalarMappable(norm=norm, cmap=cmap); sm.set_array([])
    cb = ax.figure.colorbar(sm, ax=ax, pad=0.012, fraction=0.025, aspect=12)
    cb.set_label('IG attribution', fontsize=11)
    cb.ax.tick_params(labelsize=9)
    # prediction summary as light metadata (not a title)
    p = result.prediction
    ok = 'correct' if p.correct_bin else 'incorrect'
    sub = (f'Pred {p.predicted_mz_da:.2f} / True {p.true_mz_da:.2f} m/z  |  '
           f'Δ {p.error_da:.3f} Da ({p.error_ppm:.0f} ppm)  |  bin {ok}  |  '
           f'log-prob {p.confidence:.3f}')
    ax.text(0.0, 1.015, sub, transform=ax.transAxes, fontsize=10,
            color='#444444', va='bottom')


def panel_confidence(ax, confidence, result):
    """Panel 3 — per-peak model confidence strip, bars coloured (heatmap) by
    confidence with a colourbar. No panel title (per request)."""
    masked = _masked_set(result)
    idx = np.arange(len(confidence))
    nz = confidence > 0
    norm = Normalize(vmin=0.0, vmax=1.0)
    cmap = plt.get_cmap(CONF_CMAP)
    bar_colors = [cmap(norm(confidence[i])) for i in idx[nz]]
    ax.bar(idx[nz], confidence[nz], width=0.9, color=bar_colors, zorder=3)
    for i in idx[nz]:
        if i in masked:
            ax.bar(i, confidence[i], width=0.9, facecolor='none',
                   edgecolor=MASKED_COLOR, linewidth=1.4, zorder=4)
    ax.set_xlim(-2, len(confidence) + 1)
    ax.set_ylim(0, 1.05)
    ax.set_xlabel('Peak index')
    ax.set_ylabel('Confidence')
    sm = ScalarMappable(norm=norm, cmap=cmap); sm.set_array([])
    cb = ax.figure.colorbar(sm, ax=ax, pad=0.012, fraction=0.025, aspect=12)
    cb.set_label('Confidence', fontsize=11)
    cb.ax.tick_params(labelsize=9)


def panel_pred_confidence(ax, result):
    """Panel 4a — per-peak prediction confidence for the masked group.
    No panel title (per request)."""
    pps = result.prediction.peak_predictions
    labels = [pp.annotation for pp in pps]
    # confidence as exp(log_prob)
    conf = [float(np.exp(pp.log_prob)) for pp in pps]
    ys = np.arange(len(labels))[::-1]
    colors = ['#2E75B6' if pp.correct_bin else '#D62728' for pp in pps]
    ax.barh(ys, conf, color=colors, edgecolor='white', height=0.6, zorder=3)
    for y, c, pp in zip(ys, conf, pps):
        tag = 'hit' if pp.correct_bin else 'miss'
        ax.text(c + 0.01, y, f'{c:.2f}  ({tag})', va='center', fontsize=11)
    ax.set_yticks(ys)
    ax.set_yticklabels(labels)
    ax.set_xlim(0, 1.05)
    ax.set_xlabel('Prediction confidence  exp(log-prob)')


def panel_ranked_attribution(ax, report, top_n=12):
    """Panel 4b — ranked attribution by fragment group, coloured by relationship
    category. No panel title (per request); keeps the category legend."""
    groups = report['attribution']['ranked_groups'][:top_n]
    labels = [g['annotation'] for g in groups]
    vals   = [g['attribution_for_this_group'] for g in groups]
    cats   = [g['category'] for g in groups]
    ys = np.arange(len(labels))[::-1]
    colors = [CATEGORY_COLORS.get(c, '#BDBDBD') for c in cats]
    ax.barh(ys, vals, color=colors, edgecolor='white', height=0.7, zorder=3)
    ax.set_yticks(ys)
    ax.set_yticklabels(labels)
    ax.set_xlabel('IG attribution')
    present = []
    for c in cats:
        if c not in present:
            present.append(c)
    handles = [mpatches.Patch(color=CATEGORY_COLORS.get(c, '#BDBDBD'),
                              label=CATEGORY_LABELS.get(c, c)) for c in present]
    ax.legend(handles=handles, loc='lower right', fontsize=9, ncol=1)


print('Panel renderers defined.')


In [ ]:
# ── Compose the four-panel hero figure ───────────────────────────────────────
set_publication_style()

fig = plt.figure(figsize=(14, 13))
gs = gridspec.GridSpec(4, 2, figure=fig,
                       height_ratios=[1.05, 1.0, 0.75, 1.2],
                       hspace=0.55, wspace=0.22)

ax1 = fig.add_subplot(gs[0, :])   # ion view (full width)
ax2 = fig.add_subplot(gs[1, :])   # attribution (full width)
ax3 = fig.add_subplot(gs[2, :])   # confidence strip (full width)
ax4a = fig.add_subplot(gs[3, 0])  # per-peak prediction confidence
ax4b = fig.add_subplot(gs[3, 1])  # ranked attribution

panel_ion_view(ax1, mz, intensity, annotations, result)
panel_attribution(ax2, attributions, result)
panel_confidence(ax3, confidence, result)
panel_pred_confidence(ax4a, result)
panel_ranked_attribution(ax4b, report)

fig.suptitle(title_from(report), fontsize=15, fontweight='bold', y=0.995)
fig.tight_layout(rect=[0, 0, 1, 0.98])

save_fig(fig, 'hero_010_KIEGNLIFDPNNYLPK_y6+')
plt.show()


In [ ]:
# ── Export each panel as a standalone .svg (matching the per-panel style of
#    Figures 1 & 3) ───────────────────────────────────────────────────────────
set_publication_style()

# Panel 1
fig, ax = plt.subplots(figsize=(14, 4.2))
panel_ion_view(ax, mz, intensity, annotations, result)
fig.tight_layout(); save_fig(fig, 'hero_4a_ion_view'); plt.close(fig)

# Panel 2
fig, ax = plt.subplots(figsize=(14, 4.2))
panel_attribution(ax, attributions, result)
fig.tight_layout(); save_fig(fig, 'hero_4b_ig_attribution'); plt.close(fig)

# Panel 3
fig, ax = plt.subplots(figsize=(14, 3.4))
panel_confidence(ax, confidence, result)
fig.tight_layout(); save_fig(fig, 'hero_4c_confidence_strip'); plt.close(fig)

# Panel 4a
fig, ax = plt.subplots(figsize=(7, 3.8))
panel_pred_confidence(ax, result)
fig.tight_layout(); save_fig(fig, 'hero_4d_pred_confidence'); plt.close(fig)

# Panel 4b
fig, ax = plt.subplots(figsize=(7, 4.6))
panel_ranked_attribution(ax, report)
fig.tight_layout(); save_fig(fig, 'hero_4e_ranked_attribution'); plt.close(fig)

print('\nAll panels exported to figures/4/.')


---
# Where is the model allocating its attention?

The panels above show *how much* IG-attribution each peak receives. The figures
below interpret *where* and *why* the attention goes — using the same hero data
(`y6+` masked on `KIEGNLIFDPNNYLPK`).

Key facts these figures make visible:
- **It is not merely intensity-driven** — `corr(attribution, intensity) ≈ 0.55`;
  several low-intensity peaks (`y5+[+1]`, `y9+`) are highly attributed.
- **It lands on the sequence ladder** — the most attributed fragments are the
  ladder neighbours `y7+ / y7++ / y5+` flanking the masked `y6+`, plus the
  opposite-series `b6+`.
- **It is concentrated** — Gini ≈ 0.63; the top-10 of 198 peaks hold ~33 %.

1. **Attribution in m/z space** — attention localised on the fragment ladder.
2. **Attribution vs intensity** — the model is not just following the big peaks.
3. **Attribution on the peptide backbone** — which cleavages the model leans on.
4. **Attention concentration (Lorenz / Gini)** — *how focused* the attention is.
5. **Attribution by relationship category** — *what* the attention is spent on.


In [ ]:
# ── Figure 1 — Attribution in m/z space ──────────────────────────────────────
# Real spectrum (intensity vs m/z) with each peak coloured by IG attribution.
# The ladder around the masked y6+ (y5+ ← y6+ → y7+) and the opposite-series
# b6+ are annotated, with the residue-mass gaps that connect the y-ladder.
set_publication_style()


def _find_peak(annotations, mz, intensity, target):
    """Index of the visible peak with exact annotation `target`, else None."""
    for i, a in enumerate(annotations):
        if a == target and intensity[i] > 0:
            return i
    return None


fig, ax = plt.subplots(figsize=(14, 5.0))

norm = Normalize(vmin=0.0, vmax=float(attributions.max()))
cmap = plt.get_cmap(ATTR_CMAP)
masked = set(result.masked_group.peak_indices)
vis = np.where(intensity > 0)[0]
# order so the most-attributed peaks are drawn last (on top)
for i in vis[np.argsort(attributions[vis])]:
    c = MASKED_COLOR if i in masked else cmap(norm(attributions[i]))
    lw = 2.4 if i in masked else 1.2 + 2.0 * norm(attributions[i])
    ax.vlines(mz[i], 0, intensity[i], color=c, linewidth=lw,
              zorder=5 if i in masked else 3)

ymax = intensity[vis].max()
ax.set_ylim(0, ymax * 1.30)
ax.set_xlim(mz[vis].min() - 20, mz[vis].max() + 20)
ax.set_xlabel('m/z')
ax.set_ylabel('Relative intensity')
ax.set_title('Attribution in m/z space — attention localises on the fragment ladder',
             loc='left', fontweight='bold')

sm = ScalarMappable(norm=norm, cmap=cmap); sm.set_array([])
cb = fig.colorbar(sm, ax=ax, pad=0.01, fraction=0.03, aspect=18)
cb.set_label('IG attribution', fontsize=11); cb.ax.tick_params(labelsize=9)

# annotate the ladder + opposite series
b = result.masked_group.base_idx
key_labels = {'y5+': '#1F4E79', 'y7+': '#1F4E79', 'b6+': '#E69F00'}
points = {'y6+': (mz[b], intensity[b])}
ax.annotate('y6+\n(masked)', xy=(mz[b], intensity[b]), xytext=(0, 14),
            textcoords='offset points', ha='center', va='bottom',
            fontsize=12, fontweight='bold', color=MASKED_COLOR)
for lab, col in key_labels.items():
    j = _find_peak(annotations, mz, intensity, lab)
    if j is not None:
        points[lab] = (mz[j], intensity[j])
        ax.annotate(lab, xy=(mz[j], intensity[j]), xytext=(0, 8),
                    textcoords='offset points', ha='center', va='bottom',
                    fontsize=12, fontweight='bold', color=col)

# residue-mass gap brackets along the y-ladder (y5 → y6 → y7)
ctx = report['sequence_context']
gap_y = ymax * 1.16
for left, right, info in [('y5+', 'y6+', ctx['expected_ladder_left']),
                          ('y6+', 'y7+', ctx['expected_ladder_right'])]:
    if left in points and right in points:
        x0, x1 = sorted([points[left][0], points[right][0]])
        ax.annotate('', xy=(x1, gap_y), xytext=(x0, gap_y),
                    arrowprops=dict(arrowstyle='<->', color='#444444', lw=1.3))
        ax.text((x0 + x1) / 2, gap_y + ymax * 0.015,
                f"{info['gap_residue']}  (+{info['gap_mass']:.2f} Da)",
                ha='center', va='bottom', fontsize=11, color='#444444')

ax.legend(handles=[
    mpatches.Patch(color=MASKED_COLOR, label='masked y6+'),
    mpatches.Patch(color='#1F4E79', label='y-ladder neighbour'),
    mpatches.Patch(color='#E69F00', label='opposite series (b)'),
], loc='upper left', fontsize=10)

fig.tight_layout()
save_fig(fig, 'hero_attn_1_mz_space')
plt.show()


In [ ]:
# ── Figure 2 — Attribution vs intensity ──────────────────────────────────────
# Does the model just follow the biggest peaks? Scatter of IG attribution vs
# peak intensity, coloured by ion category. Annotated, low-intensity but highly
# attributed peaks (y5+[+1], y9+) show the model is not purely intensity-driven.
set_publication_style()

vis = np.where(intensity > 0)[0]
x = intensity[vis]
y = attributions[vis]
cats = [ion_category(annotations[i]) for i in vis]
masked = set(result.masked_group.peak_indices)
r_pearson = float(np.corrcoef(x, y)[0, 1])

fig, ax = plt.subplots(figsize=(8.5, 6.5))

# faint OLS trend to show the (weak) intensity relationship
sl, ic = np.polyfit(x, y, 1)
xs = np.linspace(x.min(), x.max(), 50)
ax.plot(xs, sl * xs + ic, color='#999999', lw=1.4, ls='--', zorder=1,
        label=f'OLS trend (r = {r_pearson:.2f})')

drawn = set()
for i in vis:
    if i in masked:
        continue
    cat = ion_category(annotations[i])
    ax.scatter(intensity[i], attributions[i], s=46, alpha=0.85, zorder=3,
               color=ION_COLORS[cat], edgecolor='white', linewidth=0.5,
               label=(f'{cat}-ion' if cat != 'unannot' else 'unannotated')
               if cat not in drawn else None)
    drawn.add(cat)

# masked group peaks
for i in result.masked_group.peak_indices:
    if intensity[i] > 0:
        ax.scatter(intensity[i], attributions[i], s=90, marker='D',
                   color=MASKED_COLOR, edgecolor='black', linewidth=0.6,
                   zorder=5, label='masked y6+' if 'm' not in drawn else None)
        drawn.add('m')

# label the "smart" peaks: high attribution but low intensity, plus the top one
order = vis[np.argsort(attributions[vis])[::-1]]
intensity_rank = {idx: rk for rk, idx in enumerate(np.argsort(intensity)[::-1], 1)}
highlight = []
for i in order[:12]:
    if intensity_rank[i] >= 20:        # highly attributed yet not among biggest peaks
        highlight.append(i)
highlight = highlight[:3] + [order[0]]   # + the single most-attributed peak
for i in dict.fromkeys(highlight):
    lab = annotations[i] or 'unannot'
    ax.annotate(f'{lab}\n(int. rank {intensity_rank[i]})',
                xy=(intensity[i], attributions[i]), xytext=(12, 4),
                textcoords='offset points', fontsize=9.5, color='#333333',
                arrowprops=dict(arrowstyle='-', color='#999999', lw=0.8))

ax.set_xlabel('Relative intensity')
ax.set_ylabel('IG attribution')
ax.set_title('Attribution vs intensity — attention is not just the biggest peaks',
             loc='left', fontweight='bold')
ax.legend(loc='lower right', fontsize=10, ncol=1)

fig.tight_layout()
save_fig(fig, 'hero_attn_2_attr_vs_intensity')
plt.show()


In [ ]:
# ── Figure 3 — Attribution on the peptide backbone ───────────────────────────
# Map each attributed b/a/y fragment onto the backbone cleavage it reports on.
# b/a ions (N-terminal) sit above the backbone, y ions (C-terminal) below; each
# cleavage marker is coloured by the fragment's IG attribution. The masked y6+
# cleavage is highlighted in red — the model leans on the *adjacent* cleavages
# (y5+, y7+) and the opposite-series b6+ to reconstruct it.
import re
set_publication_style()

seq = report['spectrum']['sequence']
L = len(seq)


def parse_ion(ann):
    """('y', 7) from 'y7+' / 'y7++'; ('b', 6) from 'b6+'; a-ions -> N-terminal."""
    m = re.match(r'^([aby])(\d+)(?!\d)', ann)
    if not m:
        return None
    return m.group(1), int(m.group(2))


# aggregate max attribution per backbone bond, separately for N-term (b/a) and
# C-term (y) reporting ions. Bond x = number of N-terminal residues left of it.
bonds_b, bonds_y = {}, {}
for g in report['attribution']['ranked_groups']:
    p = parse_ion(g['annotation'])
    if not p:
        continue
    s, n = p
    attr = g['attribution_for_this_group']
    x = (L - n) if s == 'y' else n
    if not (1 <= x <= L - 1):
        continue
    d = bonds_y if s == 'y' else bonds_b
    if x not in d or attr > d[x][0]:
        d[x] = (attr, g['annotation'])

masked_x = L - int(re.match(r'y(\d+)', result.masked_group.group_key).group(1))
all_attr = [v[0] for v in list(bonds_b.values()) + list(bonds_y.values())]
norm = Normalize(vmin=0.0, vmax=max(all_attr) if all_attr else 1.0)
cmap = plt.get_cmap(ATTR_CMAP)

fig, ax = plt.subplots(figsize=(14, 4.6))

# backbone + residues
ax.plot([0.5, L + 0.5], [0, 0], color='#333333', lw=2.0, zorder=2)
for i, aa in enumerate(seq, start=1):
    ax.text(i, 0, aa, ha='center', va='center', fontsize=15, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.22', fc='white', ec='#888888', lw=1.0),
            zorder=4)

H = 0.7   # marker height above/below backbone


def _draw_bond(x, attr, label, side):
    """side = +1 (b/a, above) or -1 (y, below)."""
    is_masked = (x == masked_x and side == -1)
    col = MASKED_COLOR if is_masked else cmap(norm(attr))
    ax.plot([x + 0.5, x + 0.5], [0, side * H], color=col,
            lw=3.0 if is_masked else 2.2, zorder=3)
    ax.scatter([x + 0.5], [side * H], s=150, color=col, edgecolor='black',
               linewidth=0.6, zorder=5)
    txt = f'{label}\n(masked)' if is_masked else label
    ax.annotate(txt, xy=(x + 0.5, side * H),
                xytext=(0, 9 * side), textcoords='offset points',
                ha='center', va='bottom' if side > 0 else 'top',
                fontsize=10.5, fontweight='bold',
                color=MASKED_COLOR if is_masked else '#222222')


for x, (attr, lab) in bonds_b.items():
    _draw_bond(x, attr, lab, +1)
for x, (attr, lab) in bonds_y.items():
    _draw_bond(x, attr, lab, -1)
# the masked y6+ cleavage itself (not attributed to itself) — show its position
if masked_x not in bonds_y:
    _draw_bond(masked_x, 0.0, result.masked_group.group_key, -1)

ax.set_xlim(0.2, L + 0.8)
ax.set_ylim(-H * 2.0, H * 2.0)
ax.set_yticks([])
ax.set_xticks([])
for sp in ['left', 'right', 'top', 'bottom']:
    ax.spines[sp].set_visible(False)
ax.text(0.5, H * 1.85, 'b / a ions  (N-terminal)', fontsize=11, color='#555555', va='top')
ax.text(0.5, -H * 1.85, 'y ions  (C-terminal)', fontsize=11, color='#555555', va='bottom')
ax.set_title('Attribution on the peptide backbone — the model leans on cleavages adjacent to the masked y6+',
             loc='left', fontweight='bold')

sm = ScalarMappable(norm=norm, cmap=cmap); sm.set_array([])
cb = fig.colorbar(sm, ax=ax, pad=0.01, fraction=0.025, aspect=14)
cb.set_label('IG attribution', fontsize=11); cb.ax.tick_params(labelsize=9)

fig.tight_layout()
save_fig(fig, 'hero_attn_3_backbone_map')
plt.show()


In [ ]:
# ── Figures 4 & 5 — Attention concentration, split into two standalone panels ─
# (4) Lorenz curve + Gini: HOW concentrated the attention is.
# (5) Attribution by relationship category: WHAT the attention is spent on.
set_publication_style()

vis = np.where(intensity > 0)[0]
a = np.sort(attributions[vis])               # ascending, for the Lorenz curve
n = a.size
total = a.sum()

# Lorenz curve: cumulative share of attribution vs cumulative share of peaks
cum = np.concatenate([[0.0], np.cumsum(a) / total])
xfrac = np.arange(0, n + 1) / n
# Gini = 1 - 2 * area under the Lorenz curve (trapezoidal)
# np.trapz is deprecated (removed from the namespace later in the 2.x
# series). The project floor is numpy>=2.0.2, instanovo 1.2.2's requirement,
# so np.trapezoid is always available and no fallback is needed.
gini = 1.0 - 2.0 * np.trapezoid(cum, xfrac)

# top-k concentration (share held by the k most-attributed peaks)
desc = np.sort(attributions[vis])[::-1]
top_share = lambda k: desc[:k].sum() / total
k5, k10 = top_share(5), top_share(10)

# ── Figure 4 — Lorenz curve / Gini ───────────────────────────────────────────
fig, axL = plt.subplots(figsize=(6.5, 6.0))
axL.plot([0, 1], [0, 1], color='#999999', ls='--', lw=1.4, zorder=2,
         label='perfect equality')
axL.plot(xfrac, cum, color='#1F4E79', lw=2.4, zorder=4, label='attribution Lorenz')
axL.fill_between(xfrac, cum, xfrac, color='#1F4E79', alpha=0.12, zorder=1)

x90 = 1 - 10 / n
y90 = 1.0 - k10            # curve height just before the top-10 peaks
axL.axvline(x90, color='#D62728', ls=':', lw=1.2, zorder=3)
axL.annotate(f'top 10 peaks\nhold {k10*100:.0f}% of attribution',
             xy=(x90, y90), xytext=(0.40, 0.40), fontsize=10.5, color='#D62728',
             ha='left', va='center',
             arrowprops=dict(arrowstyle='->', color='#D62728', lw=1.1))
axL.text(0.30, 0.62, f'Gini = {gini:.2f}', transform=axL.transAxes,
         fontsize=15, fontweight='bold', color='#1F4E79', ha='center',
         bbox=dict(boxstyle='round,pad=0.35', fc='white', ec='#1F4E79', lw=1.2))
axL.set_xlim(0, 1); axL.set_ylim(0, 1)
axL.set_xlabel('Cumulative fraction of peaks\n(least to most attributed)')
axL.set_ylabel('Cumulative fraction of attribution')
axL.set_title('Attention concentration (Lorenz curve)', loc='left', fontweight='bold')
axL.legend(loc='upper left', fontsize=10)
fig.tight_layout()
save_fig(fig, 'hero_attn_4_lorenz_gini')
plt.show()

# ── Figure 5 — Attribution by relationship category ──────────────────────────
from collections import defaultdict
cat_sum = defaultdict(float)
for g in report['attribution']['ranked_groups']:
    cat_sum[g['category']] += g['attribution_for_this_group']
items = sorted(cat_sum.items(), key=lambda kv: kv[1])
labels = [CATEGORY_LABELS.get(c, c) for c, _ in items]
vals   = [v for _, v in items]
colors = [CATEGORY_COLORS.get(c, '#BDBDBD') for c, _ in items]
ys = np.arange(len(items))

fig, axR = plt.subplots(figsize=(8.0, 5.2))
axR.barh(ys, vals, color=colors, edgecolor='white', height=0.72, zorder=3)
for y, v in zip(ys, vals):
    axR.text(v + max(vals) * 0.01, y, f'{v:.2f}', va='center', fontsize=10.5)
axR.set_yticks(ys); axR.set_yticklabels(labels)
axR.set_xlim(0, max(vals) * 1.18)
axR.set_xlabel('Sum IG attribution (ranked fragment groups)')
axR.set_title('What the attention is spent on', loc='left', fontweight='bold')
fig.tight_layout()
save_fig(fig, 'hero_attn_5_category')
plt.show()

# concise quantitative summary (matches the JSON-reported values)
j = report['attribution']
print(f'Gini (computed) = {gini:.3f}   |  JSON gini = {j["gini"]:.3f}')
print(f'top-5  share = {k5*100:.1f}%  |  JSON top5_concentration  = {j["top5_concentration"]*100:.1f}%')
print(f'top-10 share = {k10*100:.1f}%  |  JSON top10_concentration = {j["top10_concentration"]*100:.1f}%')
